# Informasi vs Informasi Strategis: Dataset Global Retail Prices
Alur: (1) kenali data -> (2) cek kualitas -> (3) olah -> (4) Chart 1 (informasi) -> (5) Chart 2 (informasi strategis)

Cara pakai di Google Colab: upload `global_retail_prices_dataset.csv` lewat ikon folder di kiri,
lalu jalankan sel satu per satu (Shift + Enter).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

FILE = "global_retail_prices_dataset.csv"   # ganti kalau nama/lokasi filenya beda

# product_code dibaca sebagai teks supaya kode barcode tidak berubah jadi notasi ilmiah
df = pd.read_csv(FILE, dtype={"product_code": str})

## 1. Kenali data
Satu baris = satu catatan harga untuk satu produk, di satu toko, pada satu tanggal.

In [ ]:
print("Jumlah baris, kolom:", df.shape)
df.head()

In [ ]:
df.info()   # tipe data dan jumlah nilai yang terisi tiap kolom

In [ ]:
df.describe()   # ringkasan kolom angka: min, max, rata-rata, kuartil

## 2. Cek kualitas data
Langkah ini nentuin analisis apa yang layak dan yang tidak.

In [ ]:
# Berapa persen tiap kolom yang kosong?
kosong = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
print(kosong)

In [ ]:
# Berapa nilai unik per kolom? (kolom dengan 1 nilai unik = tidak informatif)
print(df.nunique().sort_values())

In [ ]:
# Isi kolom kategori: nilai apa saja dan berapa kali muncul
print(df["currency"].value_counts())
print()
print(df["price_is_discounted"].value_counts())   # semuanya False -> tidak bisa analisis diskon

In [ ]:
# Rentang tanggal pencatatan
print(df["date_recorded"].min(), "sampai", df["date_recorded"].max())

**Temuan kualitas data:**
- `brand` dan `country_code` kosong semua
- `price_is_discounted` isinya False semua
- `category` sebagian besar kosong
- Ada 20 mata uang, jadi harga tidak bisa dibandingkan langsung -> kita fokus ke EUR
- Kolom `city` ternyata berisi nama jaringan toko (E. Leclerc, Colruyt, dst.)

## 3. Olah data

In [ ]:
eur = df[df["currency"] == "EUR"].copy()
print("Baris EUR:", len(eur))

# Median harga per jaringan toko (hanya yang punya minimal 50 catatan)
per_toko = (
    eur.groupby("city")["price"]
    .agg(median="median", jumlah="count")
    .query("jumlah >= 50")
    .sort_values("median")
)
per_toko

In [ ]:
# Produk yang sama (product_code sama) yang tercatat di 2 jaringan toko atau lebih
e = eur.dropna(subset=["product_code"])
per_produk = e.groupby("product_code").agg(
    nama=("product_name", "first"),
    jumlah_toko=("city", "nunique"),
    harga_min=("price", "min"),
    harga_maks=("price", "max"),
)
bandingan = per_produk[per_produk["jumlah_toko"] >= 2].copy()
bandingan["selisih_persen"] = (bandingan["harga_maks"] / bandingan["harga_min"] - 1) * 100
bandingan["nama"] = bandingan["nama"].fillna(bandingan.index.to_series())
bandingan = bandingan.sort_values("selisih_persen", ascending=False)

print("Produk yang bisa dibandingkan:", len(bandingan))
bandingan.head(10)

## 4. Chart 1: INFORMASI (apa yang terjadi)
Prinsip yang dipakai: bar horizontal diurutkan, judul berpesan, satu warna sorot (sisanya abu-abu),
gridline dan border dibuang, sumbu mulai dari nol, label angka langsung di batang.

In [ ]:
ABU, SOROT, TEKS = "#B8B8B8", "#C0392B", "#333333"

fig, ax = plt.subplots(figsize=(9, 5))
warna = [SOROT if v == per_toko["median"].max() else ABU for v in per_toko["median"]]
batang = ax.barh(per_toko.index, per_toko["median"], color=warna)

for b, v, n in zip(batang, per_toko["median"], per_toko["jumlah"]):
    ax.text(v + 0.05, b.get_y() + b.get_height() / 2, f"€{v:.2f}  (n={n})",
            va="center", fontsize=10, color=TEKS)

ax.set_title("Monoprix punya median harga tertinggi di antara 6 jaringan supermarket",
             loc="left", fontsize=13, fontweight="bold", color=TEKS)
ax.set_xlabel("Median harga per catatan (EUR)")
ax.set_xlim(0, per_toko["median"].max() * 1.3)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(left=False)
fig.text(0.01, -0.02,
         "Catatan: produk yang dicatat tiap toko berbeda, jadi ini belum perbandingan apple-to-apple.\n"
         "Sumber: Global Retail & E-Commerce Product Prices (Kaggle), hanya data EUR, toko dengan >= 50 catatan.",
         fontsize=8, color="#666666", va="top")
plt.tight_layout()
plt.savefig("chart1_informasi.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Chart 2: INFORMASI STRATEGIS (keputusan apa yang diambil)
Pesan strategis: produk yang sama bisa berbeda jauh harganya antar jaringan toko,
jadi peritel perlu memantau produk dengan selisih terbesar terhadap pesaing.

In [ ]:
top = bandingan.head(10).iloc[::-1]          # 10 selisih terbesar, dibalik biar yang terbesar di atas
label = [n[:30] for n in top["nama"]]
warna2 = [SOROT if i >= len(top) - 3 else ABU for i in range(len(top))]   # sorot 3 teratas

fig, ax = plt.subplots(figsize=(9, 6))
batang = ax.barh(label, top["selisih_persen"], color=warna2)

for b, v in zip(batang, top["selisih_persen"]):
    ax.text(v + 1, b.get_y() + b.get_height() / 2, f"{v:.0f}%", va="center", fontsize=10, color=TEKS)

ax.set_title("Produk yang sama bisa selisih hingga 67% antar jaringan supermarket",
             loc="left", fontsize=13, fontweight="bold", color=TEKS)
ax.set_xlabel("Selisih harga tertinggi vs terendah (%)")
ax.set_xlim(0, top["selisih_persen"].max() * 1.15)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(left=False)

# Anotasi keputusan (teks saja, di area kosong kanan bawah, tanpa panah supaya tidak menabrak label)
ax.text(top["selisih_persen"].max() * 0.50, 2.2,
        "Keputusan: pantau & sesuaikan harga\nproduk dengan selisih terbesar\nterhadap pesaing",
        fontsize=10, color=SOROT, fontweight="bold", va="center")

fig.text(0.01, -0.02,
         f"Catatan: hanya {len(bandingan)} produk yang tercatat di 2+ jaringan toko (EUR), jadi ini indikasi, bukan kesimpulan pasti.\n"
         "Sumber: Global Retail & E-Commerce Product Prices (Kaggle).",
         fontsize=8, color="#666666", va="top")
plt.tight_layout()
plt.savefig("chart2_informasi_strategis.png", dpi=200, bbox_inches="tight")
plt.show()

## Untuk laporan
- **Informasi (Chart 1):** menjawab "apa yang terjadi" (Monoprix median-nya paling tinggi).
- **Informasi strategis (Chart 2):** menunjuk produk dengan selisih harga terbesar dan mengarah ke keputusan
  (pantau dan sesuaikan harga terhadap pesaing).
- Tulis keterbatasan: data 20 mata uang (kita pakai EUR), sampel produk pembanding kecil, produk antar toko berbeda.